# 00 · Getting the data: FDSN web services and SDS archives

**Goal.** Before any machine learning, we need waveforms and metadata. In this notebook you will:

1. see which ARGOS stations were recording on our workshop day,
2. query the station metadata (StationXML) from the EarthScope/IRIS FDSN web service,
3. download an hour of waveforms and look at the instrument response,
4. store data in an **SDS archive** (the layout used by SeisComP and by every later notebook),
5. fetch the complete day used in the rest of the workshop.

> **Case study: Aluto.** Aluto is a silicic caldera in the central Main Ethiopian Rift, host to a producing geothermal
> field. The **ARGOS** project (Aluto Regional Geophysical Observation Study) ran a temporary
> broadband network there between January 2012 and January 2014 (FDSN code **XM**, 100 Hz
> HH? channels, blank location code). Up to 12 stations recorded at once; they captured
> shallow hydrothermal seismicity, deeper magmatic events and rift faulting
> ([Wilks et al., 2017](https://doi.org/10.1016/j.jvolgeores.2017.04.003)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tesfahiwet-yemane/SeismoML/blob/main/notebooks/00_data_access.ipynb)

In [ ]:
# Running on Google Colab? This cell installs everything and gets the data (2-4 minutes).
# On your own computer it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    repo = "/content/SeismoML"
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tesfahiwet-yemane/SeismoML.git", repo], check=True)
    subprocess.run([sys.executable, f"{repo}/scripts/colab_setup.py"] + "".split(), check=True)
    sys.path.insert(0, f"{repo}/src")
    os.chdir(f"{repo}/notebooks")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import UTCDateTime

import seismoml as sml
from seismoml import data, plotting

plotting.use_style()
cfg = sml.load_config()
print(f"Workshop day : {cfg.day.date}")
print(f"Network      : {cfg.network}   channels {cfg.channels}")
print(f"Data folder  : {cfg.sds}")

## 1 · Which stations were recording?

The deployment changed over two years: stations were added, moved and removed.
The table below summarises each station epoch (from the field logs).

In [ ]:
table = data.station_table(cfg.path("stations"))
table

In [ ]:
fig = plotting.deployment_timeline(table, day=cfg.day)

In [ ]:
active = data.active_stations(table, cfg.day)
print(f"{len(active)} stations cover the whole of {cfg.day.date}:")
print(", ".join(active["station"]))

Mid-November 2012 is one of the best-instrumented periods of the experiment:
A13E had just been installed on the eastern flank (9 Oct 2012) and A02E on the northern
shore was still running (removed 22 Jan 2013), giving **11 simultaneous stations**.

## 2 · Asking the data centre: FDSN web services

[FDSN web services](https://www.fdsn.org/webservices/) are a standard set of HTTP APIs that
almost every seismological data centre exposes:

| Service | Returns |
|---|---|
| `station` | metadata (StationXML): coordinates, channels, instrument responses |
| `dataselect` | waveforms (miniSEED) |
| `availability` | what time spans exist in the archive |
| `event` | earthquake catalogues |

ObsPy wraps them in a single `Client`. ARGOS data are archived at the EarthScope (IRIS) DMC.

In [ ]:
client = data.fdsn_client(cfg.provider)
inv = client.get_stations(
    network=cfg.network,
    station="A??E",
    channel=cfg.channels,
    starttime=cfg.t0,
    endtime=cfg.t1,
    level="channel",
)
print(inv)

In [ ]:
data.inventory_table(inv)

In [ ]:
fig, _ = plotting.event_map(
    data.inventory_table(inv),
    lat=cfg.lat_range,
    lon=cfg.lon_range,
    title=f"ARGOS stations operating on {cfg.day.date}",
)

## 3 · One hour of waveforms

A `dataselect` request needs the four SEED codes and a time window. The blank location
code is written `"--"` in FDSN requests.

In [ ]:
sta = cfg.demo_station
t0 = cfg.t0 + 3600              # 01:00 UTC (04:00 local time: a quiet night hour)
t1 = t0 + 3600

st = client.get_waveforms(cfg.network, sta, "--", cfg.channels, t0, t1)
print(st)

In [ ]:
with plt.rc_context({"figure.constrained_layout.use": False}):  # ObsPy manages its own layout
    fig = st.plot(size=(1000, 450), equal_scale=False, linewidth=0.4, handle=True)

The data are in **counts**. To work in physical units we need the instrument response, which
lives in the `response`-level StationXML.

In [ ]:
inv_resp = client.get_stations(
    network=cfg.network, station=sta, channel=cfg.channels,
    starttime=t0, endtime=t1, level="response",
)
resp = inv_resp.select(channel="HHZ")[0][0][0].response
print(resp)
with plt.rc_context({"figure.constrained_layout.use": False}):
    fig = resp.plot(min_freq=0.01, output="VEL", label=f"{sta}.HHZ", sampling_rate=100.0, show=False)

In [ ]:
st_vel = st.copy().detrend("demean").taper(0.01)
st_vel.remove_response(inventory=inv_resp, output="VEL", pre_filt=(0.05, 0.1, 40, 45))
st_vel.filter("bandpass", freqmin=1, freqmax=20, zerophase=True)
z = st_vel.select(component="Z")[0]
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(z.times(), z.data * 1e6, lw=0.4, color="k")
ax.set(xlabel=f"Seconds after {t0}", ylabel="Velocity (µm/s)", title=f"{z.id}, 1–20 Hz")
ax.set_xlim(0, 3600);

**Note.** The machine-learning pickers we use later are trained on raw or simply
normalised waveforms, so they do not need response removal. Physical units matter
when we compute magnitudes (notebook 03).

## 4 · Storing data: the SDS archive

Downloading the same data again and again is slow and impolite to the data centre. We save
it locally in the **SeisComP Data Structure** (SDS):

```
<root>/<YEAR>/<NET>/<STA>/<CHA>.D/<NET>.<STA>.<LOC>.<CHA>.D.<YEAR>.<JDAY>
```

One file per channel per day. ObsPy reads such an archive with
`obspy.clients.filesystem.sds.Client`, using the same `get_waveforms` call as the FDSN client.

In [ ]:
demo_root = cfg.root / "data" / "demo_sds"
written = data.write_sds(st, demo_root)
for p in written:
    print(p.relative_to(cfg.root))

In [ ]:
sds = data.sds_client(demo_root)
st_local = sds.get_waveforms(cfg.network, sta, "*", "HHZ", t0 + 600, t0 + 660)
print(st_local)

## 5 · The full workshop day

Notebooks 01–05 use all 11 stations for the whole day. That is a few hundred MB of miniSEED.
Two ways to get it:

```bash
python scripts/fetch_data.py      # pre-packed archive from the GitHub release (fast)
python scripts/download_data.py   # straight from the FDSN service (what we did above, for 11 stations × 24 h)
```

`fetch_data.py` tries the archive first and falls back to FDSN. Both write into `data/sds`
and save the response-level StationXML to `data/inventory/XM_aluto.xml`.
The cell below runs it only if the data are not already there.

In [ ]:
import subprocess
import sys

day_files = sorted(cfg.sds.glob(f"{cfg.day.year}/{cfg.network}/*/HH?.D/*.{cfg.day.julday:03d}"))
if not day_files or not cfg.inventory_file.exists():
    subprocess.run([sys.executable, str(cfg.root / "scripts" / "fetch_data.py")], check=True)
    day_files = sorted(cfg.sds.glob(f"{cfg.day.year}/{cfg.network}/*/HH?.D/*.{cfg.day.julday:03d}"))
print(f"{len(day_files)} channel-day files, "
      f"{sum(f.stat().st_size for f in day_files) / 1e6:.0f} MB")

### Data availability

Gaps and missing channels are normal for a temporary deployment. Let's check what we
actually have before running any model.

In [ ]:
st_day = data.load_waveforms(cfg.sds, cfg.network, cfg.stations, cfg.t0, cfg.t1, channels="HHZ")
cov = data.availability(st_day, cfg.t0, cfg.t1)
fig, ax = plt.subplots(figsize=(10, 3.2))
for k, sta_ in enumerate(cfg.stations):
    for tr in st_day.select(station=sta_):
        ax.plot([tr.stats.starttime.datetime, tr.stats.endtime.datetime], [k, k],
                lw=8, solid_capstyle="butt", color="#55a868")
ax.set_yticks(range(len(cfg.stations)), cfg.stations)
ax.set_title(f"HHZ availability on {cfg.day.date}")
cov.groupby("station")["coverage"].sum().round(3).to_frame("fraction of day")

## Exercises

1. Use `client.get_events(...)` to see whether any global catalogue lists earthquakes within
   50 km of Aluto in November 2012. What does that tell you about the detection threshold of
   regional/global networks here?
2. Download one hour from a different station and compare its noise level with A10E's. Which
   stations are close to roads, villages or the geothermal plant?
3. Try `client.get_waveforms_bulk` to request several stations in one call.

**Next:** 01 · Denoising (`01_denoising.ipynb`)